In [ ]:
from typing import List, Dict

# Self-ask / self-instruct
# Self-Ask + filter
# DeepEval Synthetic Module
# Langchain SyntheticQAEvaluator

In [ ]:
# common interface

QA = Dict[str, str]  # {"question": ..., "answer": ..., "doc_id": ..., "source_text": ...}

def generate_qas_for_doc(doc_id: str, text: str, framework: str) -> List[QA]:
    ...


for doc in arxiv_docs:
    qas = generate_qas_for_doc(doc.id, doc.abstract_or_section, framework="self_instruct")
    # or framework="langchain", "deepeval", etc.


In [ ]:
# API Calls

from dotenv import load_dotenv
load_dotenv("keys.env") 
client = OpenAI()

SRC_DIR = Path("data/rag_chunks")
DST_DIR = Path("data/rag_chunks_image_summary")
DST_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "gpt-4.1-mini"  


def get_response(img_path: Path) -> str:
    
    resp = client.responses.create(
        model=MODEL,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": PROMPT},
            ],
        }],
        stream=False,
    )
    return resp.output_text.strip()

# self-ask / self-instruct

In [ ]:
# self-ask / self-instruct


SELF_INSTRUCT_PROMPT = """
You are a helpful assistant reading a research paper excerpt.

TEXT:
\"\"\"{text}\"\"\"

Generate {n_qas} diverse question-answer pairs that can be answered *directly and unambiguously* from this text alone.

Requirements:
- Cover different types: definitions, methods, motivations, comparisons, results.
- Make questions specific, not vague.
- Answers should be concise and copy or paraphrase the text.
- Return JSON as a list under key "qas", like:
  {{"qas": [{{"question": "...", "answer": "..."}}, ...]}}
"""

In [ ]:

def generate_qas_self_instruct(doc_id: str, text: str, n_qas: int = 5) -> List[QA]:
    prompt = SELF_INSTRUCT_PROMPT.format(text=text, n_qas=n_qas)
    # call your LLM here, pseudo-code:
    response = call_llm(prompt)
    data = json.loads(extract_json(response))  # you'll write a small helper to parse JSON
    qas = []
    for qa in data["qas"]:
        qas.append({
            "question": qa["question"],
            "answer": qa["answer"],
            "doc_id": doc_id,
            "source_text": text,
        })
    return qas

# Self-ask + Filter

In [ ]:
VERIFY_PROMPT = """
You are checking if a question-answer pair is fully supported by the given text.

TEXT:
\"\"\"{text}\"\"\"

QUESTION: {question}
ANSWER: {answer}

Is the answer completely supported by the text, without relying on outside knowledge?
Respond with "YES" or "NO" only.
"""

In [ ]:
def verify_qa_with_text(text: str, question: str, answer: str) -> bool:
    prompt = VERIFY_PROMPT.format(text=text, question=question, answer=answer)
    resp = call_llm(prompt).strip().upper()
    return resp.startswith("Y")

In [ ]:
def generate_qas_self_instruct_filtered(doc_id: str, text: str, n_qas: int = 5) -> List[QA]:
    raw_qas = generate_qas_self_instruct(doc_id, text, n_qas * 2)  # over-generate
    filtered = []
    for qa in raw_qas:
        if verify_qa_with_text(text, qa["question"], qa["answer"]):
            filtered.append(qa)
        if len(filtered) >= n_qas:
            break
    return filtered

# LangChain Synthetic QA

In [ ]:
from langchain.evaluation.qa import QAGenerateChain  # or synthetic modules depending on version

qa_generator = QAGenerateChain.from_llm(your_llm)

In [ ]:
def generate_qas_langchain(doc_id: str, text: str, n_qas: int = 5) -> List[QA]:
    res = qa_generator.generate(
        examples=[{"doc": text}],
        n=n_qas,  # depends on exact signature
    )
    # res might be a list of dicts like {"question": ..., "answer": ...}
    qas = []
    for qa in res:
        qas.append({
            "question": qa["question"],
            "answer": qa["answer"],
            "doc_id": doc_id,
            "source_text": text,
        })
    return qas

# DeepEval synthetic module

In [ ]:
from deepeval.synthetic import generate_qa_dataset  # name illustrative

def generate_qas_deepeval(doc_id: str, text: str, n_qas: int = 5) -> List[QA]:
    dataset = generate_qa_dataset(
        documents=[{"id": doc_id, "text": text}],
        num_questions_per_doc=n_qas,
        # plus any config like LLM, style, etc.
    )
    # dataset might be a list of {"question": ..., "answer": ..., "metadata": ...}
    qas = []
    for qa in dataset:
        qas.append({
            "question": qa["question"],
            "answer": qa["answer"],
            "doc_id": doc_id,
            "source_text": text,
        })
    return qas